In [81]:
import pandas as pd
import zipfile
import os
import json

#Project Params

params = {}
with open("params_file.json") as param_file:
    params = json.load(param_file)


def concatenate_zips(folder_path):
    all_dataframes = []

    # Loop through every file in the directory
    for filename in os.listdir(folder_path):
        if filename.endswith(".zip"):
            file_path = os.path.join(folder_path, filename)
            #Crawl through directory a
            with zipfile.ZipFile(file_path, 'r') as z:
                for internal_file in z.namelist():
                    if internal_file.endswith(".csv"):
                        with z.open(internal_file) as f:
                            df = pd.read_csv(f)
                            df['source_file'] = filename
                            all_dataframes.append(df)
    
    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)
        return master_df
    else:
        print("Empty Dataset Directory")
        return None

In [72]:
#Load main Data Table
full_dataframe = concatenate_zips(params["Dataset_Directory"] + "Flight Data")

C:\Users\cadek\AppData\Local\Temp\ipykernel_42816\1142411262.py:25: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


In [73]:
#Load Lookup Table
#BTS Lookup Tables https://transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=b0-gvzr
airport_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\T_MASTER_CORD.csv", index_col="AIRPORT_ID")
airport_lookup = airport_lookup[airport_lookup['AIRPORT_IS_LATEST'] == 1]
airport_lookup = airport_lookup[airport_lookup['AIRPORT_IS_CLOSED'] == 0]
carrier_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\Carriers.csv")
#Weather Stations https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv
station_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\Weather_Stations.csv", index_col='ICAO')
station_lookup = station_lookup[station_lookup['USAF'] != '999999']

In [22]:
full_dataframe.head()

,YEAR,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,...,DIVERTED,AIR_TIME,FLIGHTS,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,source_file
0,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5225.0,10397,1039707,30397,...,0.0,33.0,1.0,164.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
1,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5115.0,11433,1143302,31295,...,0.0,89.0,1.0,651.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
2,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5414.0,11423,1142308,31423,...,0.0,87.0,1.0,533.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
3,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N133EV,5284.0,12953,1295304,31703,...,0.0,103.0,1.0,722.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
4,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N133EV,5341.0,10994,1099402,30994,...,0.0,83.0,1.0,641.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip


In [74]:
#Find top 100 airports
dest_counts = full_dataframe['DEST_AIRPORT_ID'].value_counts().reset_index()
dest_counts.columns = ['AIRPORT_ID', 'FLIGHT_COUNT']
dest_counts.sort_values(by='FLIGHT_COUNT')

dest_counts['AIRPORT_ID'] = dest_counts['AIRPORT_ID'].astype(int)

dest_data = pd.merge(
    dest_counts, 
    airport_lookup, 
    left_on='AIRPORT_ID', 
    right_on='AIRPORT_ID',
    how='left'
)

dest_data['Tentative_Station'] = 'K' + dest_data['AIRPORT']


dest_data = pd.merge(
     dest_data, 
     station_lookup, 
     left_on='Tentative_Station', 
     right_on='ICAO',
     how='left'
 )

In [83]:
print(type(dest_counts['AIRPORT_ID'].iloc[0]))
print(type(airport_lookup.index[0]))
print(params)
print(dest_data.columns)

<class 'numpy.int64'>
<class 'numpy.int64'>
{'Dataset_Directory': 'C:\\Users\\cadek\\OneDrive\\Desktop\\CSC881 (Data Mining)\\Project\\', 'Tokens': {'Flight_Data_Token': ' ', 'Weather_Query_Token': ' '}}
Index(['AIRPORT_ID', 'FLIGHT_COUNT', 'AIRPORT_SEQ_ID', 'AIRPORT',
       'AIRPORT_STATE_CODE', 'AIRPORT_STATE_FIPS', 'Tentative_Station', 'USAF',
       'WBAN', 'STATION NAME', 'CTRY', 'STATE', 'LAT', 'LON', 'ELEV(M)',
       'BEGIN', 'END'],
      dtype='object')


In [76]:
dest_data = dest_data[dest_data['CTRY'] == 'US']
dest_data = dest_data[['AIRPORT_ID', 'FLIGHT_COUNT', 'AIRPORT_SEQ_ID', 'AIRPORT',
       'AIRPORT_STATE_CODE', 'AIRPORT_STATE_FIPS',
       'Tentative_Station', 'USAF',
       'WBAN', 'STATION NAME', 'CTRY', 'STATE', 'LAT', 'LON', 'ELEV(M)',
       'BEGIN', 'END']].drop_duplicates()

dest_data = dest_data.head(100)


In [82]:
dest_data.to_csv(params['Output_Directory'] + "top_100.csv")

KeyError: 'Output_Directory'